Câu 1: Hãy viết câu lệnh SQL để tính sự tương quan giữa A và B theo công thức sau: 

In [1]:
SELECT 
    (COUNT(*) * SUM(A * B) - SUM(A) * SUM(B)) / 
    (SQRT(COUNT(*) * SUM(A * A) - SUM(A) * SUM(A)) * SQRT(COUNT(*) * SUM(B * B) - SUM(B) * SUM(B))) AS correlation
FROM data
WHERE A IS NOT NULL AND B IS NOT NULL;

IndentationError: unexpected indent (2805874287.py, line 2)

Câu 2: Một công ty oto đang kiểm tra 3 loại mẫu mới A, B và C trong 4 ngày, và chấm điểm theo thang từ 1
đến 10 điểm cho mỗi ngày với bảng sau. Liệu có sự khác biệt đáng kể giữa các mẫu dựa trên điểm số mà
chúng nhận được trong 4 ngày thử nghiệm không? Kết quả thử nghiệm phụ thuộc vào ngày hay phụ thuộc vào
mẫu xe? Hãy chuyển đổi dữ liệu sang dạng quan hệ và thực hiện kiểm tra χ2.

In [7]:
import sqlite3
import pandas as pd
import scipy.stats as stats

# Kết nối SQLite 
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Tạo bảng dữ liệu
cursor.execute("""
    CREATE TABLE car_tests (
        Day TEXT,
        Model TEXT,
        Score REAL
    )
""")

# Dữ liệu thử nghiệm
data = [
    ("Day 1", "A", 8), ("Day 1", "B", 9), ("Day 1", "C", 7),
    ("Day 2", "A", 7.5), ("Day 2", "B", 8.5), ("Day 2", "C", 7),
    ("Day 3", "A", 6), ("Day 3", "B", 7), ("Day 3", "C", 8),
    ("Day 4", "A", 7), ("Day 4", "B", 6), ("Day 4", "C", 5)
]

# Chèn dữ liệu vào bảng
cursor.executemany("INSERT INTO car_tests VALUES (?, ?, ?)", data)
conn.commit()
# Truy vấn dữ liệu từ SQLite
query = "SELECT Day, Model, Score FROM car_tests"
df = pd.read_sql_query(query, conn)

# Chuyển dữ liệu thành bảng tần suất 
table = pd.pivot_table(df, values="Score", index="Day", columns="Model", aggfunc="sum")
# Kiểm định Chi-square
chi2_stat, p_value, dof, expected = stats.chi2_contingency(table)
print(f"Chi-square Statistic: {chi2_stat}")
print(f"P-value: {p_value}")
print(f"Degrees of Freedom: {dof}")
print("Expected Frequencies:")
print(pd.DataFrame(expected, index=table.index, columns=table.columns))
# Kiểm tra kết quả
alpha = 0.05
if p_value < alpha:
    print("Có sự khác biệt đáng kể giữa các mẫu xe.")
else:
    print("Không có đủ bằng chứng để khẳng định sự khác biệt giữa các mẫu xe.")

# Đóng kết nối SQLite
conn.close()


Chi-square Statistic: 0.826439691310499
P-value: 0.9913459315666546
Degrees of Freedom: 6
Expected Frequencies:
Model         A         B         C
Day                                
Day 1  7.953488  8.511628  7.534884
Day 2  7.622093  8.156977  7.220930
Day 3  6.959302  7.447674  6.593023
Day 4  5.965116  6.383721  5.651163
Không có đủ bằng chứng để khẳng định sự khác biệt giữa các mẫu xe.


vì giá trị p-value lớn (0.9913 > 0.05) nên không có đủ bằng chứng để bác bỏ giả thuyết Ho
nên kết quả thử nghiệm không phụ thuốc vào mẫu xe hay ngày thử nghiệm

Câu 3: Bảng flights(departure_time,...) chứa các giá trị thời gian dưới dạng số nguyên  ví dụ 830 cho 8:30AM, 1445 cho 2:45PM).Hãy chuyển dổi các giá trị này thành định dạng thời gian.

In [8]:
import sqlite3
import random

# Kết nối đến SQLite 
conn = sqlite3.connect("flights.db")
cursor = conn.cursor()

# Tạo bảng flights
cursor.execute("""
CREATE TABLE IF NOT EXISTS flights (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    departure_time_int INTEGER,
    departure_time_str TEXT
)
""")

# Xoá sạch dữ liệu cũ để mỗi lần chạy có kết quả ngẫu nhiên khác
cursor.execute("DELETE FROM flights")

# Hàm tạo thời gian ngẫu nhiên
def generate_time():
    hour = random.randint(0, 30)
    minute = random.randint(0, 28)
    return hour * 70 + minute

# Chuyển đổi sang định dạng thời gian chuẩn
def convert_to_time(time_int):
    hour = time_int // 100
    minute = time_int % 100
    return f"{hour:02d}:{minute:02d}"

# Tạo và chèn dữ liệu mới
num_samples = 100
departure_times = [(generate_time(),) for _ in range(num_samples)]
departure_times = [(t[0], convert_to_time(t[0])) for t in departure_times]

cursor.executemany("INSERT INTO flights (departure_time_int, departure_time_str) VALUES (?, ?)", departure_times)
conn.commit()

cursor.execute("SELECT id, departure_time_int, departure_time_str FROM flights LIMIT 50")
rows = cursor.fetchall()

print("ID | Departure Time (Int) | Departure Time (Str)")
print("-" * 30)
for row in rows:
    print(f"{row[0]:<3} | {row[1]:<18} | {row[2]}")

conn.close()


ID | Departure Time (Int) | Departure Time (Str)
------------------------------
201 | 1494               | 14:94
202 | 1131               | 11:31
203 | 919                | 09:19
204 | 563                | 05:63
205 | 1969               | 19:69
206 | 1891               | 18:91
207 | 1761               | 17:61
208 | 1282               | 12:82
209 | 1260               | 12:60
210 | 1632               | 16:32
211 | 1409               | 14:09
212 | 1265               | 12:65
213 | 495                | 04:95
214 | 1912               | 19:12
215 | 370                | 03:70
216 | 1406               | 14:06
217 | 1829               | 18:29
218 | 1764               | 17:64
219 | 1985               | 19:85
220 | 1691               | 16:91
221 | 841                | 08:41
222 | 1907               | 19:07
223 | 1560               | 15:60
224 | 223                | 02:23
225 | 76                 | 00:76
226 | 723                | 07:23
227 | 571                | 05:71
228 | 1683               | 16

Câu 4: Viết truy vấn SQL để tìm các ngoại lệ bằng cách sử dụng MAD. Một quy tắc chung là xem xét các giá trị ngoại lẹ lớn hơn 1.5 lần so với giá trị MAD, trong đó x là số độ lệch chuẩn mà ta coi là có ý nghĩa.

In [9]:
import sqlite3
import random
import pandas as pd

# Kết nối tới cơ sở dữ liệu SQLite 
conn = sqlite3.connect(':memory:')  
cursor = conn.cursor()

# Tạo bảng
cursor.execute('''
CREATE TABLE data (
    id INTEGER PRIMARY KEY,
    value REAL
);
''')

# Chèn dữ liệu ngẫu nhiên vào bảng
for i in range(150):
    value = round(random.uniform(0, 80), 2)  
    cursor.execute('INSERT INTO data (id, value) VALUES (?, ?)', (i + 1, value))

# Tính giá trị trung vị (Median) sử dụng ROW_NUMBER()
cursor.execute('''
WITH RankedData AS (
    SELECT value,
           ROW_NUMBER() OVER (ORDER BY value) AS row_num,
           COUNT(*) OVER () AS total_count
    FROM data
)
SELECT value
FROM RankedData
WHERE row_num = (total_count + 1) / 2;
''')
median_value = cursor.fetchone()[0]

# Tính độ lệch tuyệt đối 
cursor.execute('''
WITH RankedData AS (
    SELECT value,
           ROW_NUMBER() OVER (ORDER BY value) AS row_num,
           COUNT(*) OVER () AS total_count
    FROM data
),
deviation AS (
    SELECT value, ABS(value - (SELECT value FROM RankedData WHERE row_num = (total_count + 1) / 2)) AS abs_deviation
    FROM RankedData
)
SELECT abs_deviation FROM deviation;
''')

# Lấy tất cả các độ lệch tuyệt đối và tính MAD
deviations = [row[0] for row in cursor.fetchall()]
mad_value = sorted(deviations)[len(deviations) // 2]  # MAD là trung vị của độ lệch tuyệt đối

# In giá trị MAD
print(f"Giá trị MAD: {mad_value}")

# Tạo danh sách ngoại lệ và không phải ngoại lệ
cursor.execute('''
WITH RankedData AS (
    SELECT value,
           ROW_NUMBER() OVER (ORDER BY value) AS row_num,
           COUNT(*) OVER () AS total_count
    FROM data
),
deviation AS (
    SELECT value, ABS(value - (SELECT value FROM RankedData WHERE row_num = (total_count + 1) / 2)) AS abs_deviation
    FROM RankedData
)
SELECT id, value, ABS(value - (SELECT value FROM RankedData WHERE row_num = (total_count + 1) / 2)) AS abs_deviation
FROM data;
''')

# Lấy kết quả
rows = cursor.fetchall()

# Phân loại ngoại lệ và không phải ngoại lệ
outliers = []
non_outliers = []

for row in rows:
    id, value, abs_deviation = row
    if abs_deviation > 1.5 * mad_value:
        outliers.append((id, value))
    else:
        non_outliers.append((id, value))

# Tạo DataFrame từ các danh sách ngoại lệ và không phải ngoại lệ
df_outliers = pd.DataFrame(outliers, columns=['id', 'Ngoại lệ'])
df_non_outliers = pd.DataFrame(non_outliers, columns=['id', 'Không phải ngoại lệ'])
df_combined = pd.merge(df_outliers, df_non_outliers, left_index=True, right_index=True, how='outer')

# in toàn bộ DataFrame
pd.set_option('display.max_rows', None)  # Hiển thị tất cả các dòng
pd.set_option('display.max_columns', None)  # Hiển thị tất cả các cột
pd.set_option('display.width', None)  # Không giới hạn chiều rộng hiển thị
print("\nDataFrame chứa giá trị ngoại lệ và không phải ngoại lệ:")
print(df_combined)

# Đóng kết nối
conn.close()


Giá trị MAD: 20.189999999999998

DataFrame chứa giá trị ngoại lệ và không phải ngoại lệ:
      id_x  Ngoại lệ  id_y  Không phải ngoại lệ
0      4.0     74.30     1                40.19
1      5.0      5.47     2                14.18
2      9.0     72.89     3                55.08
3     10.0      6.46     6                51.13
4     12.0     72.99     7                32.79
5     14.0      4.37     8                23.67
6     18.0     74.71    11                68.18
7     19.0     79.72    13                69.41
8     25.0     75.09    15                16.51
9     26.0      8.11    16                33.29
10    29.0      5.46    17                41.04
11    33.0     72.74    20                18.74
12    35.0     71.72    21                24.28
13    51.0      0.28    22                49.09
14    58.0      8.93    23                36.90
15    61.0     78.40    24                28.83
16    65.0     76.88    27                46.35
17    67.0     73.79    28                29.97

Câu 5 : hãy xác định liệu hai người trong bảng Patient(last_name, weight, height) có phải là một người hay không bằng cách sử dụng khảng cách kết hợp BOOLEAN trên "last_name" và " weight"

In [6]:
import sqlite3
import random
import string

# Kết nối tới cơ sở dữ liệu SQLite 
conn = sqlite3.connect(':memory:')  
cursor = conn.cursor()

# Tạo bảng Patient
cursor.execute('''
CREATE TABLE Patient (
    id INTEGER PRIMARY KEY,
    last_name TEXT,
    weight REAL,
    height REAL
);
''')

# Hàm tạo tên ngẫu nhiên tên từ A-Z 
def generate_random_name():
    length = random.randint(4, 8)  
    return ''.join(random.choices(string.ascii_uppercase, k=length))  # Chọn ngẫu nhiên các chữ cái từ A-Z

# Chèn dữ liệu ngẫu nhiên vào bảng Patient
for i in range(15):
    last_name = generate_random_name()  # Tạo tên ngẫu nhiên
    weight = round(random.uniform(20, 150), 1)  # Số ngẫu nhiên từ 50 đến 100
    height = round(random.uniform(120, 140), 1)  # Số ngẫu nhiên từ 150 đến 190
    cursor.execute('INSERT INTO Patient (last_name, weight, height) VALUES (?, ?, ?)', 
                   (last_name, weight, height))

cursor.execute('SELECT * FROM Patient')
patients = cursor.fetchall()

# Kiểm tra sự tương đồng giữa các cặp bệnh nhân dựa trên last_name và weight
matches = []

for i in range(len(patients)):
    for j in range(i + 1, len(patients)):
        patient1 = patients[i]
        patient2 = patients[j]
        
        # Xác định liệu hai người có phải là một người dựa trên last_name và weight
        last_name_match = (patient1[1] == patient2[1])  # So sánh last_name
        weight_match = (patient1[2] == patient2[2])  # So sánh weight
        
        # Nếu cả hai đều giống nhau, coi là một người
        if last_name_match and weight_match:
            matches.append((patient1, patient2, True))  # Hai người là một người
        else:
            matches.append((patient1, patient2, False))  # Hai người không phải là một người

# In kết quả
print("Kết quả so sánh:")
for match in matches:
    patient1, patient2, is_same_person = match
    print(f"Patient 1 (ID: {patient1[0]}): {patient1[1]}, {patient1[2]}kg, {patient1[3]}cm")
    print(f"Patient 2 (ID: {patient2[0]}): {patient2[1]}, {patient2[2]}kg, {patient2[3]}cm")
    print(f"Is same person: {is_same_person}")
    print()

# Đóng kết nối
conn.close()


Kết quả so sánh:
Patient 1 (ID: 1): NUIC, 20.2kg, 125.3cm
Patient 2 (ID: 2): RLRBTKZ, 148.3kg, 123.6cm
Is same person: False

Patient 1 (ID: 1): NUIC, 20.2kg, 125.3cm
Patient 2 (ID: 3): GZCFPVYK, 38.3kg, 127.7cm
Is same person: False

Patient 1 (ID: 1): NUIC, 20.2kg, 125.3cm
Patient 2 (ID: 4): AWCUIG, 26.9kg, 129.7cm
Is same person: False

Patient 1 (ID: 1): NUIC, 20.2kg, 125.3cm
Patient 2 (ID: 5): FUSWLKB, 55.4kg, 130.6cm
Is same person: False

Patient 1 (ID: 1): NUIC, 20.2kg, 125.3cm
Patient 2 (ID: 6): OEFWB, 122.6kg, 120.3cm
Is same person: False

Patient 1 (ID: 1): NUIC, 20.2kg, 125.3cm
Patient 2 (ID: 7): ESUT, 96.6kg, 123.8cm
Is same person: False

Patient 1 (ID: 1): NUIC, 20.2kg, 125.3cm
Patient 2 (ID: 8): ALUTXMO, 58.1kg, 133.0cm
Is same person: False

Patient 1 (ID: 1): NUIC, 20.2kg, 125.3cm
Patient 2 (ID: 9): UINBQ, 110.7kg, 127.7cm
Is same person: False

Patient 1 (ID: 1): NUIC, 20.2kg, 125.3cm
Patient 2 (ID: 10): GGFV, 67.5kg, 133.1cm
Is same person: False

Patient 1 (ID: 1)